# Tracking Hyperparameter Tuning with MLflow

MLflow provides a powerful framework for hyperparameter tuning that allows systematically explore the hyperparameter space and find the best model.

### Prerequisites: Set up MLflow and Optuna

In [2]:
# pip install optuna

### Create a new experiment

In [5]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("Hyper parameter Tuning Experiment")

2026/09/22 10:38:31 INFO mlflow.tracking.fluent: Experiment with name 'Hyper parameter Tuning Experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1790051911315, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1790051911315, lifecycle_stage='active', name='Hyper parameter Tuning Experiment', tags={}, trace_location=None, workspace='default'>

### Prepare Data

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing

X, y = fetch_california_housing(return_X_y=True)
X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=0)

### Define the objective function

In Optuna, a study is a single optimization task, representing the entire hyperparameter tuning session consisting of multiple trials. A trial is a single execution of the objective function, namely, training a model with a single combination of hyperparameters.

In MLflow, this structure is represented by a <b>parent run and child runs.</b> A parent run is a run that contains all the child runs for different trials. The parent-child relationship allows us to keep track of each trial during the hyperparameter tuning process and group them nicely in the MLflow UI.

First, let's define the objective function that is executed for each trial. To log the parameters, metrics, and model file, we use MLflow's API inside the objective function. An MLflow run is created with the nested=True flag to indicate it is a child run.


##### For, <b>mlflow.sklearn.log_model(regressor_obj, name="model")</b> :
* MLflow Action: Packages the trained Scikit-Learn Python object into a standardized MLflow Model directory inside the Artifact Store (your local directory or remote cloud bucket).

* What MLflow creates behind the scenes: MLflow does not just save a .pkl file; it generates a complete deployable folder named model/ containing:

  1. <b>model.pkl:</b> The serialized model weights.
  2. <b>MLmodel:</b> A YAML configuration file stating the model flavor (python_function and sklearn), serialization format, and MLflow version.
  3. <b>conda.yaml</b> and requirements.txt: The exact Python dependencies required to reproduce and serve this model on any other machine.

In [10]:
import mlflow
import optuna
import sklearn


def objective(trial):
    # Setting nested=True will create a child run under the parent run.
    
    # if an outer parent run is already open, calling mlflow.start_run() without nested=True causes MLflow to raise an error, it also set a hidden tag on this new run: mlflow.parentRunId = <parent_run_id>
    
    # as child_run: This captures the ActiveRun Python object into a variable. It holds metadata properties—most importantly child_run.info.run_id, which is the run's unique 32-character hexadecimal fingerprint (e.g., a1b2c3d4...).
    
    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}") as child_run:
        rf_max_depth = trial.suggest_int("rf_max_depth", 2, 32)
        rf_n_estimators = trial.suggest_int("rf_n_estimators", 50, 300, step=10)
        rf_max_features = trial.suggest_float("rf_max_features", 0.2, 1.0)
        params = {
            "max_depth": rf_max_depth,
            "n_estimators": rf_n_estimators,
            "max_features": rf_max_features,
        }
        # Log current trial's parameters
        mlflow.log_params(params)

        regressor_obj = sklearn.ensemble.RandomForestRegressor(**params)
        regressor_obj.fit(X_train, y_train)

        y_pred = regressor_obj.predict(X_val)
        error = sklearn.metrics.mean_squared_error(y_val, y_pred)
        
        # Log current trial's error metric, MLflow Action: Logs numerical evaluation metrics to the server.
        # metrics also accept a step argument (e.g., step=epoch in training loops)
        mlflow.log_metrics({"error": error})

        # Log the model file
        
        # MLflow detected the skops library. Newer MLflow versions prioritize skops over standard cloudpickle when available. skops has a strict zero-trust security policy by default. Telling MLflow to serialize the model using standard cloudpickle instead of skops by passing serialization_format="cloudpickle"
        mlflow.sklearn.log_model(regressor_obj, name="model", serialization_format="cloudpickle")
        # Make it easy to retrieve the best-performing child run later
        trial.set_user_attr("run_id", child_run.info.run_id)
        return error

### Run the hyperparameter tuning study
let's run the hyperparameter tuning study using Optuna. We create a parent run named "study" and log the best trial's parameters and metrics there. In hyperparameter tuning, algorithms like Optuna generate dozens or hundreds of trials. If we log every trial as an independent top-level entity, our experiment list turns into an unreadable flat table of clutter.

This block of code runs the entire search inside an MLflow Parent Run (named "study")

In [11]:
# Create a parent run that contains all child runs for different trials
with mlflow.start_run(run_name="study") as run: # Creates a new top-level run record in your SQLite backend database and marks it as active on the current execution thread.
    # Log the experiment settings
    n_trials = 30
    mlflow.log_param("n_trials", n_trials)

    # Optuna runs 30 times. In each iteration, it enters your objective function, which spins up a nested child run (nested=True), trains a model, logs metrics and artifacts to MLflow, and attaches its child_run.info.run_id to Optuna's user_attrs
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)

    # Log the best trial and its run ID, Takes the dictionary of parameters from Optuna's best trial (e.g., {"max_depth": 14, "n_estimators": 220, "max_features": 0.8}) and attaches them directly to the parent run.
    mlflow.log_params(study.best_trial.params)
    
    # Logs the single best numerical outcome (lowest MSE) achieved across all 30 trials as a top-level metric on the parent run.
    mlflow.log_metrics({"best_error": study.best_value})
    
    # Save a pointer to the winning model artifact
    if best_run_id := study.best_trial.user_attrs.get("run_id"):
        mlflow.log_param("best_child_run_id", best_run_id)
        
        
    # # we can rewrite the pervious 2 line code this way for better understanding
    # # Step A: Look into the best trial's backpack dictionary (user_attrs)
    # # and try to extract the value stored under the key "run_id"
    # best_run_id = study.best_trial.user_attrs.get("run_id")

    # # Step B: If it exists and is not None or empty
    # if best_run_id is not None:
    #     # Step C: Log it into MLflow on the parent run
    #     mlflow.log_param("best_child_run_id", best_run_id)

[I 2026-09-22 13:33:25,699] A new study created in memory with name: no-name-d5cd6924-5ccc-4784-a99a-01a590499c72
2026/09/22 13:33:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[I 2026-09-22 13:33:35,873] Trial 0 finished with value: 0.27030688143921294 and parameters: {'rf_max_depth': 13, 'rf_n_estimators': 60, 'rf_max_features': 0.2686469235766715}. Best is trial 0 with value: 0.27030688143921294.


🏃 View run trial_0 at: http://127.0.0.1:5001/#/experiments/2/runs/79877bc397d2421196bc4d41e0b32384
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2


2026/09/22 13:34:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[I 2026-09-22 13:34:06,386] Trial 1 finished with value: 0.2657364289029584 and parameters: {'rf_max_depth': 23, 'rf_n_estimators': 300, 'rf_max_features': 0.9156960627936339}. Best is trial 1 with value: 0.2657364289029584.


🏃 View run trial_1 at: http://127.0.0.1:5001/#/experiments/2/runs/998755c1bf754c509044ded6b9fc6ffb
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2


2026/09/22 13:34:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

🏃 View run trial_28 at: http://127.0.0.1:5001/#/experiments/2/runs/527bc1da5d114cd48ee16fecd26caa49
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2


[I 2026-09-22 13:40:27,497] Trial 28 finished with value: 0.252887147592774 and parameters: {'rf_max_depth': 22, 'rf_n_estimators': 260, 'rf_max_features': 0.5129581825099229}. Best is trial 22 with value: 0.24523611145436175.
2026/09/22 13:40:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run trial_29 at: http://127.0.0.1:5001/#/experiments/2/runs/f26856f8d08e4091880e1857951d351e
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2


[I 2026-09-22 13:40:45,390] Trial 29 finished with value: 0.24484269171738277 and parameters: {'rf_max_depth': 30, 'rf_n_estimators': 270, 'rf_max_features': 0.41626867788635186}. Best is trial 29 with value: 0.24484269171738277.


🏃 View run study at: http://127.0.0.1:5001/#/experiments/2/runs/8960e4fa02554c06a77d79299dc83bd5
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2


### Register Your Best Model
Once you identified the best trial, you can register the model into MLflow Model Registry for promoting it to production.

1. <b>runs:/:</b> A scheme prefix telling MLflow: "Look inside the artifacts of an experiment run".
2. <b><run_id>:</b> The 32-character hexadecimal ID of the specific trial or training run.
3. <b><artifact_path>:</b> The subfolder name where you saved the model (for example, "model" if you called mlflow.sklearn.log_model(..., name="model") or artifact_path="model").

##### If wanna find out manually the model path, follow this step:
* In the MLflow UI:

    1. Open your browser to [http://127.0.0.1:5001](http://127.0.0.1:5001).

    2. In the left sidebar, click on Experiments and select your experiment (Hyper parameter Tuning Experiment).

    3. You will see two tabs at the top: Runs and Models.

    4. In the Runs tab, locate your parent run (study). Look at the parameters column: you will see best_child_run_id set to f26856f8d08e4091880e1857951d351e.

    5. In the Models tab, every model trained across all trials is listed. The trial with the lowest error (0.2448...) is trial_29, which matches that exact Run ID. Click on it, and you will see the exact page you quoted where the status shows Not registered.

In [12]:
# URI stands for 'Uniform Resource Identifier'.

import mlflow

# 1. Connect to tracking server
mlflow.set_tracking_uri("http://127.0.0.1:5001")

# 2. Point to the winning trial's model artifact
winning_model_uri = "runs:/f26856f8d08e4091880e1857951d351e/model"

# 3. Register it
registered_model = mlflow.register_model(
    model_uri=winning_model_uri,
    name="housing-price-predictor",
)

print(
    f"Success! Registered: {registered_model.name} Version: {registered_model.version}"
)

Successfully registered model 'housing-price-predictor'.
2026/09/22 15:00:44 WARNING mlflow.tracking._model_registry.fluent: Run with id f26856f8d08e4091880e1857951d351e has no artifacts at artifact path 'model', registering model based on models:/m-017a1a5e3da14ebaaef26654984f8465 instead
2026/09/22 15:00:44 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: housing-price-predictor, version 1
Created version '1' of model 'housing-price-predictor'.


Success! Registered: housing-price-predictor Version: 1


#### What should be in production code? (Fully Automatic Selection)
In production, never hardcode a run ID string by hand. The code automatically picks the best trial from Optuna, registers it, and tags it with a production alias (such as @champion). Downstream inference services only load that alias.

In [ ]:
# The Training & Auto-Registration Script

import mlflow
import optuna
from mlflow import MlflowClient

mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("Hyper parameter Tuning Experiment")

# 1. Run tuning
with mlflow.start_run(run_name="study") as parent_run:
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=30)

    # 2. Extract winning run ID programmatically from Optuna
    best_run_id = study.best_trial.user_attrs["run_id"]
    mlflow.log_param("best_child_run_id", best_run_id)

# 3. Automatically register the winning model
model_name = "housing-price-predictor"
model_version = mlflow.register_model(
    model_uri=f"runs:/{best_run_id}/model", name=model_name
)

# 4. Assign the 'champion' alias so production knows this is the active model
client = MlflowClient()
client.set_registered_model_alias(
    name=model_name, alias="champion", version=model_version.version
)

print(
    f"Model v{model_version.version} automatically selected and marked as @champion!"
)

In [ ]:
# The Production Serving Code (FastAPI, Docker, or Batch). Inference service does not need to know which trial won or what the run ID was. It simply requests the @champion

import mlflow
import pandas as pd

mlflow.set_tracking_uri("http://127.0.0.1:5001")

# Always loads the current winning production model automatically
champion_model = mlflow.pyfunc.load_model(
    "models:/housing-price-predictor@champion"
)

# Predict on live input
predictions = champion_model.predict(new_data)